<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/a8/%D0%9B%D0%9E%D0%93%D0%9E_%D0%A8%D0%90%D0%94.png" width=500px/>
    <font>Python 2025</font><br/>
    <br/>
    <br/>
    <b style="font-size: 2em">Protobuf & Flatbuf</b><br/>
    <br/>
    <font>Ислам Абасов</font><br/>
</center>

#### Введение
* Краткое объяснение, зачем нужны сериализация и десериализация данных.
* Обзор популярных форматов сериализации (JSON, XML, Protobuf и др.)
#### Protocol Buffers (Protobuf)
* Что такое Protocol Buffers:
* Структура и синтаксис Protobuf:
* Работа с Protobuf в Python:
* Сценарии использования и ограничения:
#### Fletbuf 
* Что такое Fletbuf:
* Работа с Fletbuf в Python:
* Особенности и отличия от Protobuf:

## Введение

_Сериализация_ — это процесс преобразования объекта в последовательность байтов для сохранения его состояния или передачи по сети. Это позволяет сохранить объекты в файлы или передавать их между различными компонентами систем.

_Десериализация_ — это обратный процесс, заключающийся в преобразовании последовательности байтов обратно в объект, восстанавливая его состояние. Таким образом, десериализация позволяет восстановить объект из сохраненного или переданного представления.

В целом, сериализация и десериализация используются для обмена данными между системами, хранения объектов и работы с удаленными вызовами процедур (RPC).

Сериализация и десериализация необходимы по нескольким причинам:

1. **Сохранение состояния**: Сериализация позволяет сохранять состояние объектов в файлы или базы данных. Это полезно, когда необходимо восстановить данные после перезапуска приложения или серверов.

2. **Передача данных**: При обмене данными между различными модулями, компонентами или системами (например, клиент-серверные приложения) сериализация позволяет формировать поток данных для передачи по сети.

3. **Кросс-платформенность**: С использованием сериализации можно передавать объекты между различными платформами или языками программирования, обеспечивая взаимодействие между разными системами.

4. **Кэширование**: Сохранение сериализованных объектов в кэш позволяет быстро восстанавливать их состояние без необходимости повторного вычисления или загрузки из источников данных.

5. **Удаленные вызовы процедур (RPC)**: В распределенных системах, где функции одного сервера могут быть использованы другим сервером или клиентом, сериализация помогает формировать вызовы процедур с передачей необходимых данных.

[Сравнение форматов серриализации на GO](https://habr.com/ru/companies/avito/articles/786756/)

#### Основные текстовые форматы сериализации
- **JSON (JavaScript Object Notation):**
  - Популярность: простой и читаемый человеком формат, который широко используется в разработке веб-приложений.
  - Прочтение и написание: идеально подходит для взаимодействия между клиентом и сервером.

- **XML (Extensible Markup Language):**
  - Гибкость и мощность, подходит для сложных структур данных и метаданных.
  - Недостатки: объемный и может быть сложным в обработке из-за громоздкой структуры.

#### Альтернативные бинарные форматы сериализации
- **pickle**
  - опасно для передачи данных по сети
- **numpy**
- **protobuf**
- **flatbuf**
- **parquet**
- **MessagePack:**
- **Apache Avro:**
- **CBOR (Concise Binary Object Representation):**

### Protocol Buffers (Protobuf)

#### Что такое Protocol Buffers:
- **Определение:** Protocol Buffers (Protobuf) — это гибкий и эффективный механизм сериализации структурированных данных, разработанный Google.
- **Преимущества:**
  - Компактность и эффективность в сравнении с текстовыми форматами, такими как JSON и XML.
  - Независимость от языка программирования и платформы.
  - Поддержка версионности схем.

## [Использование protobuf в python](https://protobuf.dev/getting-started/pythontutorial/)

In [3]:
%%writefile proto/proto_exmple.proto

syntax = "proto3";

package tutorial;

import "google/protobuf/timestamp.proto";

message TeacherAttr {
    optional string subject = 1;
    optional int32 experience_years = 2;
}

message DiggerAttr {
    optional string company = 1;
    optional bool certified = 2;
}

message Phone {
    enum Type {
        MOBILE = 0;
        HOME = 1;
        WORK = 2;
    }

    Type type = 1;
    reserved 2;
    reserved "number";
}

message Person {
    string name = 1;
    google.protobuf.Timestamp birthday = 2;
    reserved 3;
    repeated string email = 4;
    map<string, int64> add_attrs = 5;
    oneof job_attrs {
        TeacherAttr teacher_attr = 6;
        DiggerAttr digger_attr = 7;
    }
    Phone phone = 8;
    int64 age = 9;

    reserved "old_name";
}

Overwriting proto/proto_exmple.proto


In [4]:
!protoc --python_out=. proto/proto_exmple.proto --cpp_out=.

In [8]:
from proto.proto_exmple_pb2 import Person, Phone, TeacherAttr
from google.protobuf.timestamp_pb2 import Timestamp
import datetime as dt

ts = Timestamp()
ts.FromDatetime(dt.datetime(1995, 5, 20, 12, 0, tzinfo=dt.timezone.utc))

phone = Phone(type=Phone.MOBILE, number="1234567890")
teacher = TeacherAttr(subject="Math", experience_years=5)

p = Person(name="Alice", birthday=ts, phone=phone, age=29,)
p.email.extend(["alice@example.com", "alice@work.com"])
p.add_attrs["score"] = 100
p.add_attrs["visits"] = 42
p.teacher_attr.CopyFrom(teacher)

print(p)
b = p.SerializeToString()
print(b)

name: "Alice"
birthday {
  seconds: 800971200
}
email: "alice@example.com"
email: "alice@work.com"
add_attrs {
  key: "visits"
  value: 42
}
add_attrs {
  key: "score"
  value: 100
}
teacher_attr {
  subject: "Math"
  experience_years: 5
}
phone {
  number: "1234567890"
}
age: 29

b'\n\x05Alice\x12\x06\x08\xc0\xb3\xf7\xfd\x02"\x11alice@example.com"\x0ealice@work.com*\n\n\x06visits\x10**\t\n\x05score\x10d2\x08\n\x04Math\x10\x05B\x0c\x12\n1234567890H\x1d'


In [9]:
p2 = Person()
p2.ParseFromString(b)
print(p2)

name: "Alice"
birthday {
  seconds: 800971200
}
email: "alice@example.com"
email: "alice@work.com"
add_attrs {
  key: "visits"
  value: 42
}
add_attrs {
  key: "score"
  value: 100
}
teacher_attr {
  subject: "Math"
  experience_years: 5
}
phone {
  number: "1234567890"
}
age: 29



In [10]:
from google.protobuf.json_format import MessageToJson
j = MessageToJson(p)
print(j)

{
  "name": "Alice",
  "birthday": "1995-05-20T12:00:00Z",
  "email": [
    "alice@example.com",
    "alice@work.com"
  ],
  "addAttrs": {
    "score": "100",
    "visits": "42"
  },
  "teacherAttr": {
    "subject": "Math",
    "experienceYears": 5
  },
  "phone": {
    "number": "1234567890"
  },
  "age": "29"
}


In [11]:
from google.protobuf.json_format import Parse

p3 = Person()
Parse(j, p3)
print(p3.WhichOneof("job_attrs"))
p3

teacher_attr


name: "Alice"
birthday {
  seconds: 800971200
}
email: "alice@example.com"
email: "alice@work.com"
add_attrs {
  key: "visits"
  value: 42
}
add_attrs {
  key: "score"
  value: 100
}
teacher_attr {
  subject: "Math"
  experience_years: 5
}
phone {
  number: "1234567890"
}
age: 29

### Какие есть свойства у protobuf
* компактная двоичная сериализация
* схема-ориентированность и строгие типы
* кросс-платформенность
* обратня и прямая совместимость
    * старые клиенты игнорируют неизвестные поля
    * новые - могут добавлять без поломок дополнительные поля
* не потокобезопасны
* требуется выравнивание версии компилятора и рантайма
* для прочтения сообщения, требуется его полное чтение

#### Сценарии использования и ограничения:

- **Сценарии использования:**
  - **Сетевые коммуникации:** Protobuf широко используется для обмена данными между сервисами в распределенных системах благодаря эффективности и компактности.
  - **Хранение данных:** Подходит для сериализации структурированных данных в файлы или базы данных.
  - **Мобильные приложения:** Используется для передачи данных между клиентами и серверами, обеспечивая минимум трафика из-за своей компактности.

- **Ограничения:**
  - **Человеконечитаемость:** В отличие от JSON или XML, данные в Protobuf не предназначены для чтения человеком, что может усложнить отладку без специальных инструментов.
  - **Отсутствие самодокументированности:** В отличие от XML, структура данных не содержит явных метаданных, поэтому необходимо всегда иметь под рукой `.proto` файл для понимания структуры сообщения.
  - **Порог входа:** Использование Protobuf подразумевает изучение его синтаксиса и настройку инфраструктуры для компиляции `.proto` файлов, что может быть сложнее для новичков по сравнению с использованием JSON.

## [Flatbuf](https://flatbuffers.dev/)

#### Что такое Fletbuf:
- **Определение:** Fletbuf (часто называемый FlatBuffers) — это библиотека для сериализации данных, разработанная в Google. Она позволяет быстро производить сериализацию и десериализацию данных без генерации дополнительных промежуточных копий.
- **Преимущества:**
  - Высокая производительность при чтении данных, так как не требует их разбора до использования.
  - Поддержка множественных языков программирования, что облегчает интеграцию в кросс-языковых проектах.
  - Поддержка версионности для обращения к более старым и более новым версиям схем данных

In [12]:
%%writefile proto/flat_example.fbs
namespace tutorial;

enum PhoneType:byte { MOBILE, HOMWE, WORK }

table Phone{
    type:PhoneType;
    phone_number: string;
}

table Person {
    name: string;
    age: int;
    old_field: int(deprecated);
    email:[string];
    phone: Phone;
}

Overwriting proto/flat_example.fbs


In [13]:
!flatc --python proto/flat_example.fbs

In [15]:
import flatbuffers
from tutorial import Person, Phone, PhoneType

builder = flatbuffers.Builder(0)

phone_number = builder.CreateString('111111')
name = builder.CreateString("<NAME>")

Phone.Start(builder)
Phone.AddPhoneNumber(builder, phone_number)
Phone.AddType(builder, PhoneType.PhoneType.MOBILE)
phone = Phone.End(builder)

emails = [builder.CreateString('<EMAIL_1>'), builder.CreateString('<EMAIL_2>')]

Person.StartEmailVector(builder, len(emails))
for email in emails:
    builder.PrependUOffsetTRelative(email)
emails_vector = builder.EndVector()

Person.Start(builder)
Person.AddName(builder, name)
Person.AddAge(builder, 25)
Person.AddEmail(builder, emails_vector)
Person.AddPhone(builder, phone)
person = Person.End(builder)

builder.Finish(person)

ser = builder.Output()
print(ser)


bytearray(b'\x14\x00\x00\x00\x00\x00\x0e\x00\x14\x00\x10\x00\x0c\x00\x00\x00\x08\x00\x04\x00\x0e\x00\x00\x00D\x00\x00\x00\x0c\x00\x00\x00\x19\x00\x00\x00@\x00\x00\x00\x02\x00\x00\x00\x08\x00\x00\x00\x14\x00\x00\x00\t\x00\x00\x00<EMAIL_2>\x00\x00\x00\t\x00\x00\x00<EMAIL_1>\x00\x00\x00\x08\x00\x08\x00\x00\x00\x04\x00\x08\x00\x00\x00\x10\x00\x00\x00\x06\x00\x00\x00<NAME>\x00\x00\x06\x00\x00\x00111111\x00\x00')


In [16]:
p = Person.Person.GetRootAs(ser)

print(p.Age())
print(p.Name())

25
b'<NAME>'


Разница между protobuf и flatbuf
https://capnproto.org/news/2014-06-17-capnproto-flatbuffers-sbe.html